# X-ray luminosity interpolator

This notebook create an interpolator function for the cooling curves obtained from magneto-thermal simulations for different values of the initial magnetic field.
The interpolator serves as a function that, given the age and the initial magnetic field of a neutron star, gives as output the X-ray thermal luminosity at that age. 
Since we have only a small set of magneto-thermal cooling curves, we need to interpolate between them to obtain the X-ray luminosity for any given age and initial magnetic field.

In [ ]:
# import libraries
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scipy.integrate as integrate
from scipy import interpolate
from scipy.interpolate import UnivariateSpline
import pickle
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

import pypopsyn.simulator.basics.constants as const
from pypopsyn.simulator.config_simulator import cfg
import utilities.plot_settings

## Load the results from the magneto-thermal simulations

In [ ]:
df_B12 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e12_Btor1e13.csv",
    delimiter=",",
    header=[0],
)
df_B12.head()

In [ ]:
df_B13 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e13_Btor1e14.csv",
    delimiter=",",
    header=[0],
)
df_B13.head()

In [ ]:
df_B14 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e14_Btor1e15.csv",
    delimiter=",",
    header=[0],
)
df_B14.head()

In [ ]:
df_B15 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_B15.head()

In [ ]:
df_B5e15 = pd.read_csv(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip5e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_B5e15.head()

In [ ]:
t12 = df_B12["t[yr]"].to_numpy().astype(float)
t13 = df_B13["t[yr]"].to_numpy().astype(float)
t14 = df_B14["t[yr]"].to_numpy().astype(float)
t15 = df_B15["t[yr]"].to_numpy().astype(float)
t5e15 = df_B5e15["t[yr]"].to_numpy().astype(float)
L12 = df_B12["L[erg/s]"].to_numpy().astype(float)
L13 = df_B13["L[erg/s]"].to_numpy().astype(float)
L14 = df_B14["L[erg/s]"].to_numpy().astype(float)
L15 = df_B15["L[erg/s]"].to_numpy().astype(float)
L5e15 = df_B5e15["L[erg/s]"].to_numpy().astype(float)

In [ ]:
# Define an array with the log10 of the initial magnetic field values for the different cooling curves.
log_B0 = np.array([12, 13, 14, 15, np.log10(5.0e15)])

# Define initial magnetic fields where to evaluate the interpolated cooling curves.
# Note that this is needed now to set the right colors.
log_B0_eval = np.linspace(11.0, 17.0, 30)

# Combine the arrays to find the global min and max values.
combined_values = np.concatenate([log_B0, log_B0_eval])
vmin, vmax = combined_values.min(), combined_values.max()

# Create a colormap and normalize it.
cmap = plt.cm.viridis
norm = Normalize(vmin=vmin, vmax=vmax)

Plot the simulated cooling curves as a function of time and colorcode them according to their initial magnetic field strength.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1.0e29, 1.0e37)
ax.set_xlabel(r"Time [yr]")
ax.set_ylabel(r"$L_{X}$ [erg s$^{-1}$]")

ax.plot(
    t12,
    L12,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    t13,
    L13,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
ax.plot(
    t14,
    L14,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    t15,
    L15,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
ax.plot(
    t5e15,
    L5e15,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

NOTE: the sharp decrease of luminosity in the first hundreds, thousands of years is due to neutrinos produced in Urca processes in the neutron star interiors (where neutrons transforms into protons and viceversa) when the neutron star is still very hot. These neutrinos are free to escape the star end carry out a lot of energy so that the temperature drops faster. After that, cooling mainly proceed by the emission of photons from the surface and it is slower. This is more visible in low-magnetic field stars because there the magnetic field dissipation in the crust is not very effective in keeping the star hot. Indeed for those stars the temperature drops as if they were not magnetized and the colling curves are also very similar. For high magnetic fields the fast drop at the beginning is a bit masked by the fact that the field dissipates and heats up the neutron star crust keeping high the temperature for longer time.

## Construct the interpolator function

As the length of the various cooling curves is different, let's first interpolate them using a univariate spline on the same time grid in [yr].
In the plot, the interpolated curves are shown with dashed lines.

In [ ]:
time_grid = np.logspace(0.0, 6.0, 100)

L12_interpolator = interpolate.InterpolatedUnivariateSpline(t12, L12, k=1)
L13_interpolator = interpolate.InterpolatedUnivariateSpline(t13, L13, k=1)
L14_interpolator = interpolate.InterpolatedUnivariateSpline(t14, L14, k=1)
L15_interpolator = interpolate.InterpolatedUnivariateSpline(t15, L15, k=1)
L5e15_interpolator = interpolate.InterpolatedUnivariateSpline(t5e15, L5e15, k=1)

L12_interp = L12_interpolator(time_grid)
L13_interp = L13_interpolator(time_grid)
L14_interp = L14_interpolator(time_grid)
L15_interp = L15_interpolator(time_grid)
L5e15_interp = L5e15_interpolator(time_grid)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1.0e29, 1.0e37)
ax.set_xlabel(r"Time [yr]")
ax.set_ylabel(r"$L_{X}$ [erg s$^{-1}$]")

ax.plot(
    t12,
    L12,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    t13,
    L13,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
ax.plot(
    t14,
    L14,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    t15,
    L15,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
ax.plot(
    t5e15,
    L5e15,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L12_interp,
    linestyle="--",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L13_interp,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L14_interp,
    linestyle="--",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L15_interp,
    linestyle="--",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L5e15_interp,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)

# Create a legend for the line styles.
style_handles = [
    Line2D(
        [0], [0], color="black", linestyle="-", linewidth=4, label="Original"
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=4,
        label="Interpolated",
    ),
]
plt.legend(handles=style_handles, frameon=False, loc=0)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

In [ ]:
# Create a grid of initial magnetic fields and stack together all the cooling curves.
B0 = 10**log_B0
L_t_stack = np.vstack(
    (L12_interp, L13_interp, L14_interp, L15_interp, L5e15_interp)
).T

# Define the minimum and maximum age in [yr] and the minimum and maximum initial magnetic field in [G].
time_range = np.array([0.0, 1.0e8])
B0_range = np.array([1.0e11, 1.0e17])

Lx_interpolator = interpolate.RectBivariateSpline(
    time_grid,
    B0,
    L_t_stack,
    bbox=[time_range[0], time_range[1], B0_range[0], B0_range[1]],
    kx=1,
    ky=1,
)

In [ ]:
# Save the interpolator function and try to import it again to see if it works.
with open(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/interpolator_Lx.pkl",
    "wb",
) as f:
    pickle.dump(Lx_interpolator, f)

with open(
    "../../pypopsyn/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/interpolator_Lx.pkl",
    "rb",
) as f:
    Lx_interpolator_import = pickle.load(f)

In [ ]:
# Define a grid of times and initial magnetic fields at which we evaluate the interpolated cooling curves.
t_eval = np.logspace(0.0, 6.0, 500)
B0_eval = 10**log_B0_eval

Lt_interp = Lx_interpolator_import(t_eval, B0_eval)
print(Lt_interp.shape)

Plot the resulting interpolation for a range of values.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(1.0e29, 1.0e37)
ax.set_xlabel(r"Time [yr]")
ax.set_ylabel(r"$L_{X}$ [erg s$^{-1}$]")

ax.plot(
    time_grid,
    L12_interp,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L13_interp,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L14_interp,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L15_interp,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
ax.plot(
    time_grid,
    L5e15_interp,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)

for i in range(len(B0_eval)):
    ax.plot(
        t_eval,
        Lt_interp[:, i],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B0_eval[i])),
        rasterized=True,
        alpha=0.5,
    )

# Create a legend for the line styles.
style_handles = [
    Line2D(
        [0], [0], color="black", linestyle="-", linewidth=4, label="Original"
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=4,
        label="Interpolated",
    ),
]
plt.legend(handles=style_handles, frameon=False, loc=0)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\log_{10}(B_0 \, {\rm[G]})$")

plt.grid()

In [ ]:
# Try to evaluate the X-ray luminosity for a set of random values of the age and the initial magnetic field.
t_eval_test = np.array([1.0e3, 1.0e2])
B0_eval_test = np.logspace(13.0, 15, 2)

Lt_interp_test = Lx_interpolator_import.ev(t_eval_test, B0_eval_test)
print(Lt_interp_test)